# Retention Copilot: run the agent and the evaluation on Kaggle

Settings (right panel): Accelerator = *GPU T4 x2*, *Internet = On*. Run the cells ONE AT A TIME (never 'Run All').

1-2 install and start the model. 3 gets the code. 4 smoke test. 5-6 DEVELOPMENT runs on the dev pool (used to tune prompts and choose a model). 7 the FINAL evaluation on the held-out pool: run it only when the prompts are frozen. 8 package results.

If a cell errors, paste the error back.

In [ ]:
GITHUB_USER = "Randeep-Sidhu"
MODEL = "granite4:micro"

# Install Ollama (Internet must be On)
!apt-get install -y -q zstd pciutils lshw > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# Start the server in the background and download the model
import subprocess, time
server = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(10)
!ollama pull {MODEL}

In [ ]:
# Get the code (works whether or not the repo has a nested folder), install extras, run the offline tests
import os, pathlib
%cd /kaggle/working
!rm -rf retention-copilot
!git clone https://github.com/{GITHUB_USER}/Retention-copilot.git retention-copilot
root = next(p.parent.parent for p in pathlib.Path("retention-copilot").rglob("src/agent.py"))
os.chdir(root)
print("project root:", root)
!pip install -q langgraph langchain-ollama fastembed pytest
!python -m pytest -q 2>&1 | tail -5

In [ ]:
# Smoke test (development customers): 3 through the full agent, 2 through the no-policy baseline
!python -m src.agent --customer auto --n 3 --config rag+verify --no-persist --model {MODEL}
!python -m src.agent --customer auto --n 2 --config none --no-persist --model {MODEL}

In [ ]:
# DEVELOPMENT run 1: 16 dev customers x 3 configurations (~5 min). --fresh clears earlier dev runs.
!python -m src.evaluate --pool dev --n 16 --model {MODEL} --configs none,rag,rag+verify --fresh
!sed -n '/## Results/,/## Grounding/p' reports/dev_eval_granite4-micro.md

In [ ]:
# DEVELOPMENT run 2: IBM's larger Granite model (downloads ~19 GB; uses both GPUs). Same 16 customers.
BIG = "granite4:small-h"
!ollama pull {BIG}
!python -m src.evaluate --pool dev --n 16 --model {BIG} --configs none,rag,rag+verify
!sed -n '/## Results/,/## Grounding/p' reports/dev_eval_granite4-small-h.md

In [ ]:
# FINAL evaluation on the held-out pool: 6 configurations x 48 customers (resumable). Run ONLY when told the prompts are frozen.
!python -m src.evaluate --pool final --n 48 --model {MODEL}

In [ ]:
# Package the results, then download results.zip from the Output panel (/kaggle/working)
import zipfile, pathlib
root = pathlib.Path.cwd()
with zipfile.ZipFile("/kaggle/working/results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for p in [*root.glob("db/*_runs.db"), *root.glob("reports/**/*")]:
        if p.is_file():
            z.write(p, p.relative_to(root))
print("results.zip written")